# 01 · Analisi esplorativa (EDA)

Obiettivo: **capire i dati prima di modellarli**. Alla fine di questo notebook dovresti saper rispondere a:

1. Quanti guasti ci sono e di che tipo? (sbilanciamento delle classi)
2. Il tipo di macchina influisce sul tasso di guasto?
3. Quali variabili separano le lavorazioni regolari dai guasti?
4. Perché le feature fisiche (`power_w`, `temp_diff_k`, `wear_x_torque`) aiutano il modello?

> Prerequisito: aver eseguito `pmc data` (o `pmc setup`). Le funzioni usate qui sono in `src/pmcopilot/analysis/eda.py`: il notebook le esplora passo per passo, la CLI le usa per generare i report.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from pmcopilot.analysis import eda
from pmcopilot.analysis.style import apply_style
from pmcopilot.config import get_settings

apply_style()
settings = get_settings()
df = eda.load_eda_frame(settings)   # SQL (SQLite) -> pandas, + feature fisiche (numpy)
df.head()

## 1. Struttura e sbilanciamento

`df.info()` mostra tipi e valori mancanti; `value_counts(normalize=True)` le proporzioni.
Nota quanto è piccola la classe positiva: è il motivo per cui nel training useremo **PR-AUC** e non l'accuracy.

In [ ]:
df.info()
print()
print(df["machine_failure"].value_counts(normalize=True).rename("proporzione"))

In [ ]:
# Un modello che dice SEMPRE "nessun guasto" che accuracy avrebbe?
accuracy_banale = 1 - df["machine_failure"].mean()
print(f"Accuracy del modello banale: {accuracy_banale:.1%}  <- alta ma inutile!")

## 2. Guasti per tipo di macchina e per modalità

`groupby` è l'equivalente pandas di `GROUP BY` in SQL.

In [ ]:
eda.failure_rate_by_type(df)

In [ ]:
eda.failure_mode_counts(df)

In [ ]:
# Lo stesso risultato, scritto direttamente in SQL sul database:
from contextlib import closing

from pmcopilot.data.database import connect_readonly

query = '''
SELECT m.machine_type, COUNT(*) AS n, SUM(r.machine_failure) AS guasti,
       ROUND(AVG(r.machine_failure), 4) AS tasso
FROM sensor_readings r JOIN machines m USING (machine_id)
GROUP BY m.machine_type ORDER BY tasso DESC
'''
with closing(connect_readonly(settings.db_path)) as conn:
    display(pd.read_sql_query(query, conn))

## 3. Distribuzioni: guasti vs lavorazioni regolari

Gli istogrammi usano `density=True`: confrontiamo le **forme**, altrimenti i pochi guasti sarebbero invisibili.
Osserva `tool_wear_min` (i guasti si concentrano oltre ~200 min) e `power_w` (guasti nelle code).

In [ ]:
path = settings.figures_dir / "eda_distributions.png"
eda.plot_distributions(df, path)
Image(filename=str(path))

## 4. Il grafico chiave: velocità vs coppia

Le linee tratteggiate sono le **iso-potenze** a 3500 W e 9000 W, calcolate con numpy: `coppia = P·60 / (2π·rpm)`.
I guasti arancioni in basso a destra stanno sotto la curva dei 3500 W: sono i guasti PWF.
Un albero decisionale con solo `rpm` e `torque_nm` deve approssimare questa curva con tanti "gradini";
con `power_w` gli basta un taglio.

In [ ]:
path = settings.figures_dir / "eda_speed_torque.png"
eda.plot_speed_torque(df, path)
Image(filename=str(path))

## 5. Correlazioni

Velocità e coppia sono fortemente anticorrelate (a potenza circa costante). La correlazione lineare con il target è debole per tutte le variabili: le relazioni sono **non lineari** (soglie), un altro indizio che i modelli ad albero funzioneranno meglio della regressione logistica.

In [ ]:
path = settings.figures_dir / "eda_correlation.png"
eda.plot_correlation(df, path)
Image(filename=str(path))

## 6. Verifica di una regola con le maschere booleane

Controlliamo la regola HDF del manuale: *delta termico < 8,6 K **e** velocità < 1380 rpm*.

In [ ]:
zona_hdf = (df["temp_diff_k"] < 8.6) & (df["rpm"] < 1380)
print(f"Lavorazioni nella zona HDF: {zona_hdf.sum()}")
print(f"Di queste, con guasto HDF: {df.loc[zona_hdf, 'hdf'].mean():.0%}")
print(f"Guasti HDF fuori dalla zona: {df.loc[~zona_hdf, 'hdf'].sum()}")

## Esercizi

1. Calcola con pandas il tasso di guasto per **linea** e per **giorno** (`df["ts"].dt.date`). Ci sono giorni anomali?
2. Verifica la regola OSF: per ogni tipo di macchina, quanti guasti OSF hanno `wear_x_torque` sopra la soglia di `domain.OSF_LIMIT`?
3. Disegna un istogramma di `power_w` con due linee verticali a 3500 e 9000 W (`ax.axvline`).
4. Scrivi in SQL la query che restituisce, per ogni macchina, la percentuale di lavorazioni con usura > 200 min.
5. **Domanda da colloquio**: perché non possiamo usare le colonne `twf`, `hdf`, ... come feature del modello?